In [1]:
import pandas as pd

df = pd.read_csv("/home/dell/ML-Learning/datasets/online_retail_II_features.csv")
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
print(df.shape)
df.head()

(1021452, 28)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancelled,has_customer_id,...,year_month,day_part,is_holiday_season,total_price,items_in_invoice,total_units_in_invoice,invoice_total_value,order_value_tier,is_bulk_order,is_domestic
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,True,...,2009-12,morning,True,83.4,8,166,505.3,bulk,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,True,...,2009-12,morning,True,81.0,8,166,505.3,bulk,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,True,...,2009-12,morning,True,81.0,8,166,505.3,bulk,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,True,...,2009-12,morning,True,100.8,8,166,505.3,bulk,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,True,...,2009-12,morning,True,30.0,8,166,505.3,bulk,False,True


In [2]:
summary_overview = pd.DataFrame({
    "metric": [
        "total_transactions", "total_columns", "date_range_start", "date_range_end",
        "unique_customers", "unique_products", "unique_countries", "unique_invoices",
        "cancelled_orders", "guest_orders", "domestic_orders_pct"
    ],
    "value": [
        len(df), df.shape[1],
        df["invoicedate"].min(), df["invoicedate"].max(),
        df["customer_id"].nunique(), df["stockcode"].nunique(),
        df["country"].nunique(), df["invoice"].nunique(),
        int(df["is_cancelled"].sum()), int((~df["has_customer_id"]).sum()),
        round(df["is_domestic"].mean() * 100, 2)
    ]
})
summary_overview

,metric,value
0,total_transactions,1021452
1,total_columns,28
2,date_range_start,2009-12-01 07:45:00
3,date_range_end,2011-12-09 12:50:00
4,unique_customers,5875
5,unique_products,4918
6,unique_countries,43
7,unique_invoices,46961
8,cancelled_orders,18014
9,guest_orders,227266


In [3]:
revenue_by_country = (
    df[~df["is_cancelled"]]
    .groupby("country")
    .agg(
        total_revenue=("total_price", "sum"),
        avg_order_value=("invoice_total_value", "mean"),
        num_orders=("invoice", "nunique"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
    .round(2)
)
revenue_by_country

,total_revenue,avg_order_value,num_orders
country,,,
United Kingdom,16801618.52,1097.54,36188
EIRE,623414.16,1606.01,581
Netherlands,549773.41,6134.64,216
Germany,383289.00,858.45,753
France,311090.29,867.62,598
Australia,167800.01,7052.43,89
Spain,97766.75,880.96,144
Switzerland,94024.59,2469.87,85
Sweden,86319.14,1309.25,99


In [4]:
top_products = (
    df[~df["is_cancelled"]]
    .groupby(["stockcode", "description"])
    .agg(
        total_quantity_sold=("quantity", "sum"),
        total_revenue=("total_price", "sum"),
        num_orders=("invoice", "nunique"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
    .round(2)
)
top_products

,,total_quantity_sold,total_revenue,num_orders
stockcode,description,,,
22423,REGENCY CAKESTAND 3 TIER,26478,330590.32,3918
85123A,WHITE HANGING HEART T-LIGHT HOLDER,94142,257546.20,5356
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1
47566,PARTY BUNTING,28200,148318.28,2674
85099B,JUMBO BAG RED RETROSPOT,77280,145961.83,3245
84879,ASSORTED COLOUR BIRD ORNAMENT,80082,129324.49,2807
22086,PAPER CHAIN KIT 50'S CHRISTMAS,35084,117760.29,2018
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247
79321,CHILLI LIGHTS,15841,80540.88,1135


In [5]:
monthly_summary = (
    df[~df["is_cancelled"]]
    .groupby("year_month")
    .agg(
        total_revenue=("total_price", "sum"),
        num_orders=("invoice", "nunique"),
        avg_order_value=("invoice_total_value", "mean"),
    )
    .round(2)
)
monthly_summary

,total_revenue,num_orders,avg_order_value
year_month,,,
2009-12,798118.53,1666,1599.11
2010-01,612365.50,1049,1329.45
2010-02,537926.70,1189,746.04
2010-03,760880.74,1646,914.13
2010-04,646525.36,1436,778.04
2010-05,642428.49,1483,781.71
2010-06,697382.32,1615,845.77
2010-07,633079.42,1507,685.89
2010-08,674192.89,1402,840.60


In [6]:
order_timing = (
    df[~df["is_cancelled"]]
    .groupby("day_of_week")
    .agg(
        num_orders=("invoice", "nunique"),
        total_revenue=("total_price", "sum"),
    )
    .round(2)
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)
order_timing

,num_orders,total_revenue
day_of_week,,
Monday,6245,3415968.18
Tuesday,7147,3880421.41
Wednesday,7115,3374748.09
Thursday,8167,4010544.94
Friday,5992,3167800.98
Saturday,30,9803.05
Sunday,4824,1785174.11


In [7]:
order_tier_summary = (
    df[~df["is_cancelled"]]
    .drop_duplicates(subset="invoice")   # one row per invoice, since tier is invoice-level
    .groupby("order_value_tier")
    .agg(
        num_orders=("invoice", "count"),
        total_revenue=("invoice_total_value", "sum"),
    )
    .round(2)
)
order_tier_summary

,num_orders,total_revenue
order_value_tier,,
bulk,9448,12836733.60
large,24431,6567984.64
medium,3699,220248.71
small,1942,19493.82


In [8]:
top_customers = (
    df[df["has_customer_id"] & ~df["is_cancelled"]]
    .groupby("customer_id")
    .agg(
        total_spent=("total_price", "sum"),
        num_orders=("invoice", "nunique"),
        num_countries=("country", "nunique"),
    )
    .sort_values("total_spent", ascending=False)
    .head(10)
    .round(2)
)
top_customers

,total_spent,num_orders,num_countries
customer_id,,,
18102.0,580987.04,145,1
14646.0,526751.52,145,1
14156.0,303069.88,144,1
14911.0,272252.79,373,1
17450.0,244784.25,51,1
13694.0,195640.69,143,1
17511.0,172132.87,60,1
16446.0,168472.50,2,1
16684.0,147142.77,55,1


In [11]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [9]:
with pd.ExcelWriter("/home/dell/ML-Learning/datasets/summary_tables.xlsx") as writer:
    summary_overview.to_excel(writer, sheet_name="overview", index=False)
    revenue_by_country.to_excel(writer, sheet_name="revenue_by_country")
    top_products.to_excel(writer, sheet_name="top_products")
    monthly_summary.to_excel(writer, sheet_name="monthly_summary")
    order_timing.to_excel(writer, sheet_name="order_timing")
    order_tier_summary.to_excel(writer, sheet_name="order_value_tiers")
    top_customers.to_excel(writer, sheet_name="top_customers")

print("All summary tables saved to summary_tables.xlsx")

All summary tables saved to summary_tables.xlsx
